# 3차시: Multi-Agent (AutoGen) 실습

AutoGen 기반 멀티 에이전트 시스템 구축 실습 노트북입니다.

## 목차
8. Tool Use (도구 사용)
9. FunctionTool과 고급 도구 패턴
10. Termination 조건
11. 에이전트 상태 저장과 복원
12. 디버깅과 모니터링
13. 통합 실습: 코드 리뷰 에이전트 팀
14. 실전 팁과 베스트 프랙티스

## 환경 설정

AutoGen 패키지 설치 및 Groq API 키 설정

In [7]:
# AutoGen 설치 (최초 1회)
# !pip install -U "autogen-agentchat" "autogen-ext[openai]"

In [8]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

api_key = os.environ.get("GROQ_API_KEY")
print(f"API key loaded: {'Yes' if api_key else 'No'}")

API key loaded: Yes


In [10]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelFamily

model_client = OpenAIChatCompletionClient(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
    model_info={
        "max_tokens": 4096,
        "temperature": 0.7,
        "vision": False,
        "function_calling": True,
        "family": ModelFamily.UNKNOWN,
        "structured_output": True,
        "json_output": True,
    }
)

---
## 8. Tool Use (도구 사용)

일반 Python 함수를 도구로 등록하여 Agent가 외부 기능을 호출할 수 있게 합니다.
- 함수의 **타입 힌트**와 **독스트링**이 자동으로 도구 스키마로 변환됩니다.
- `reflect_on_tool_use=True`로 설정하면 도구 결과를 자연어로 정리합니다.

### 8-1. 기본 도구 정의 및 Agent 등록

In [11]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console


# 도구 1: 날씨 조회 (시뮬레이션)
async def get_weather(city: str) -> str:
    """주어진 도시의 현재 날씨를 조회합니다."""
    weather_data = {
        "서울": "맑음, 22°C, 습도 45%",
        "부산": "흐림, 19°C, 습도 70%",
        "제주": "비, 17°C, 습도 85%",
    }
    return weather_data.get(city, f"{city}: 날씨 데이터 없음")


# 도구 2: 수학 계산
async def calculate(expression: str) -> str:
    """수학 표현식을 안전하게 계산합니다. 예: '234 * 567', '(10 + 20) / 3'"""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "오류: 허용되지 않는 문자가 포함되어 있습니다."
    try:
        result = eval(expression)
        return f"계산 결과: {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


# Agent에 도구 등록
tool_agent = AssistantAgent(
    name="tool_agent",
    model_client=model_client,
    tools=[get_weather, calculate],
    system_message="도구를 사용하여 사용자의 요청을 처리하세요. 한국어로 답변하세요.",
    reflect_on_tool_use=True,  # 도구 결과를 자연어로 정리
)

# 실행
await Console(tool_agent.run_stream(task="서울과 부산의 날씨를 알려주고, 234 * 567을 계산해줘"))

---------- TextMessage (user) ----------
서울과 부산의 날씨를 알려주고, 234 * 567을 계산해줘
---------- ToolCallRequestEvent (tool_agent) ----------
[FunctionCall(id='ksx8wzrns', arguments='{"city":"서울"}', name='get_weather'), FunctionCall(id='5zdc1qqfk', arguments='{"city":"부새"}', name='get_weather'), FunctionCall(id='wx5z3abw2', arguments='{"expression":"234 * 567"}', name='calculate')]
---------- ToolCallExecutionEvent (tool_agent) ----------
[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='ksx8wzrns', is_error=False), FunctionExecutionResult(content='부새: 날씨 데이터 없음', name='get_weather', call_id='5zdc1qqfk', is_error=False), FunctionExecutionResult(content='계산 결과: 132678', name='calculate', call_id='wx5z3abw2', is_error=False)]
---------- TextMessage (tool_agent) ----------
서울의 날씨는 맑으며 기온은 22°C, 습도는 45%입니다. 부산의 날씨 정보는 제공되지 않았습니다.

234 * 567의 계산 결과는 132,678입니다.


TaskResult(messages=[TextMessage(id='8684972c-38b9-4337-805a-96012d85f2c0', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 44, 870831, tzinfo=datetime.timezone.utc), content='서울과 부산의 날씨를 알려주고, 234 * 567을 계산해줘', type='TextMessage'), ToolCallRequestEvent(id='12970cae-157e-4eb3-9e29-c72fa4454b32', source='tool_agent', models_usage=RequestUsage(prompt_tokens=823, completion_tokens=97), metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 45, 524582, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='ksx8wzrns', arguments='{"city":"서울"}', name='get_weather'), FunctionCall(id='5zdc1qqfk', arguments='{"city":"부새"}', name='get_weather'), FunctionCall(id='wx5z3abw2', arguments='{"expression":"234 * 567"}', name='calculate')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='27891a20-a3c0-4d30-902d-2ed9294b304b', source='tool_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 45, 528235,

### 8-2. `reflect_on_tool_use` 비교

`reflect_on_tool_use=False`일 때는 도구 결과가 그대로 반환되고, `True`일 때는 LLM이 자연어로 정리합니다.

In [12]:
# reflect_on_tool_use=False → 도구 결과 그대로 반환
raw_agent = AssistantAgent(
    name="raw_agent",
    model_client=model_client,
    tools=[get_weather],
    system_message="도구를 사용하여 날씨를 알려주세요.",
    reflect_on_tool_use=False,  # 도구 결과 그대로
)

print("=== reflect_on_tool_use=False ===")
result_raw = await Console(raw_agent.run_stream(task="제주 날씨 알려줘"))

print("\n=== reflect_on_tool_use=True ===")
# 위에서 만든 tool_agent는 reflect_on_tool_use=True
result_reflect = await Console(tool_agent.run_stream(task="제주 날씨 알려줘"))

=== reflect_on_tool_use=False ===
---------- TextMessage (user) ----------
제주 날씨 알려줘
---------- ToolCallRequestEvent (raw_agent) ----------
[FunctionCall(id='5xpc18dkj', arguments='{"city":"제주"}', name='get_weather')]
---------- ToolCallExecutionEvent (raw_agent) ----------
[FunctionExecutionResult(content='비, 17°C, 습도 85%', name='get_weather', call_id='5xpc18dkj', is_error=False)]
---------- ToolCallSummaryMessage (raw_agent) ----------
비, 17°C, 습도 85%

=== reflect_on_tool_use=True ===
---------- TextMessage (user) ----------
제주 날씨 알려줘
---------- ToolCallRequestEvent (tool_agent) ----------
[FunctionCall(id='54a03706d', arguments='{"city":"제주"}', name='get_weather')]
---------- ToolCallExecutionEvent (tool_agent) ----------
[FunctionExecutionResult(content='비, 17°C, 습도 85%', name='get_weather', call_id='54a03706d', is_error=False)]
---------- TextMessage (tool_agent) ----------
제주도의 현재 날씨는 비가 내리고 있으며 기온은 17°C, 습도는 85%입니다.


### 8-3. Team에서 도구 전담 Agent 구성

여러 Agent가 각자 다른 도구를 담당하는 패턴입니다. SelectorGroupChat에서 LLM이 적합한 Agent를 선택합니다.

In [13]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination


# 도구별 전담 Agent
weather_agent = AssistantAgent(
    "WeatherAgent",
    description="날씨 정보를 조회하는 에이전트. 날씨 관련 질문에 응답.",
    model_client=model_client,
    tools=[get_weather],
    system_message="날씨 도구를 사용하여 정확한 날씨 정보를 제공하세요.",
    reflect_on_tool_use=True,
)

calc_agent = AssistantAgent(
    "CalcAgent",
    description="수학 계산을 수행하는 에이전트. 계산 관련 질문에 응답.",
    model_client=model_client,
    tools=[calculate],
    system_message="계산 도구를 사용하여 정확한 결과를 제공하세요.",
    reflect_on_tool_use=True,
)

summarizer = AssistantAgent(
    "Summarizer",
    description="다른 에이전트의 결과를 종합하여 최종 답변을 작성하는 에이전트.",
    model_client=model_client,
    system_message="다른 에이전트의 결과를 종합하여 최종 답변을 작성하세요. 완료 시 'DONE'이라고 적으세요.",
)

termination = TextMentionTermination("DONE") | MaxMessageTermination(10)

team = SelectorGroupChat(
    [weather_agent, calc_agent, summarizer],
    model_client=model_client,
    termination_condition=termination,
)

await Console(team.run_stream(
    task="서울 날씨를 알려주고, 화씨로 변환해줘. (공식: °F = °C × 9/5 + 32)"
))

---------- TextMessage (user) ----------
서울 날씨를 알려주고, 화씨로 변환해줘. (공식: °F = °C × 9/5 + 32)
---------- ToolCallRequestEvent (WeatherAgent) ----------
[FunctionCall(id='0bxwrm7tt', arguments='{"city":"서울"}', name='get_weather')]
---------- ToolCallExecutionEvent (WeatherAgent) ----------
[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='0bxwrm7tt', is_error=False)]
---------- TextMessage (WeatherAgent) ----------
서울의 현재 날씨는 맑음이며, 기온은 22°C, 습도는 45%입니다.

화씨로 변환하면, 

(22 × 9/5) + 32 = 71.6°F

입니다.
---------- TextMessage (Summarizer) ----------
DONE


TaskResult(messages=[TextMessage(id='2b7dcd33-f823-4bcb-a40d-fac9e9a057cc', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 47, 180437, tzinfo=datetime.timezone.utc), content='서울 날씨를 알려주고, 화씨로 변환해줘. (공식: °F = °C × 9/5 + 32)', type='TextMessage'), ToolCallRequestEvent(id='ea219a58-e610-447f-bbbd-250812ac2431', source='WeatherAgent', models_usage=RequestUsage(prompt_tokens=764, completion_tokens=36), metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 47, 721617, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='0bxwrm7tt', arguments='{"city":"서울"}', name='get_weather')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='e1393cb9-9cb1-4413-ae98-160b3252698d', source='WeatherAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 47, 722789, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='0bxwrm7tt', is_error=Fals

---
## 9. FunctionTool과 고급 도구 패턴

`FunctionTool` 클래스를 사용하면 도구의 이름과 설명을 커스터마이즈할 수 있습니다.
또한 도구 에러 처리, 도구 파이프라인 패턴을 살펴봅니다.

### 9-1. FunctionTool로 이름/설명 커스터마이즈

In [ ]:
from autogen_core.tools import FunctionTool


# 원본 함수
async def translate(text: str, target_lang: str) -> str:
    """텍스트를 번역합니다."""
    # 실제로는 번역 API 호출
    translations = {
        "en": {"안녕하세요": "Hello", "감사합니다": "Thank you"},
        "ja": {"안녕하세요": "こんにちは", "감사합니다": "ありがとうございます"},
    }
    lang_data = translations.get(target_lang, {})
    result = lang_data.get(text, f"[{target_lang}] {text} (번역됨)")
    return result


# FunctionTool로 래핑 - 이름과 설명 커스터마이즈
translate_tool = FunctionTool(
    func=translate,
    name="translate_text",
    description="텍스트를 지정된 언어로 번역합니다. "
                "target_lang은 ISO 639-1 코드를 사용합니다: en(영어), ja(일본어), zh(중국어).",
)

# 스키마 확인
print(f"도구 이름: {translate_tool.name}")
print(f"도구 설명: {translate_tool.description}")
print(f"스키마: {translate_tool.schema}")

도구 이름: translate_text
도구 설명: 텍스트를 지정된 언어로 번역합니다. target_lang은 ISO 639-1 코드를 사용합니다: en(영어), ja(일본어), zh(중국어).
스키마: {'name': 'translate_text', 'description': '텍스트를 지정된 언어로 번역합니다. target_lang은 ISO 639-1 코드를 사용합니다: en(영어), ja(일본어), zh(중국어).', 'parameters': {'type': 'object', 'properties': {'text': {'description': 'text', 'title': 'Text', 'type': 'string'}, 'target_lang': {'description': 'target_lang', 'title': 'Target Lang', 'type': 'string'}}, 'required': ['text', 'target_lang'], 'additionalProperties': False}, 'strict': False}


In [15]:
# FunctionTool을 사용하는 Agent
translator_agent = AssistantAgent(
    name="translator",
    model_client=model_client,
    tools=[translate_tool],
    system_message="번역 도구를 사용하여 사용자의 번역 요청을 처리하세요.",
    reflect_on_tool_use=True,
)

await Console(translator_agent.run_stream(task="'안녕하세요'를 영어와 일본어로 번역해줘"))

---------- TextMessage (user) ----------
'안녕하세요'를 영어와 일본어로 번역해줘
---------- ToolCallRequestEvent (translator) ----------
[FunctionCall(id='pceyyyxe5', arguments='{"target_lang":"en","text":"안녕하세요"}', name='translate_text'), FunctionCall(id='qpysqx3n6', arguments='{"target_lang":"ja","text":"안녕하세요"}', name='translate_text')]
---------- ToolCallExecutionEvent (translator) ----------
[FunctionExecutionResult(content='Hello', name='translate_text', call_id='pceyyyxe5', is_error=False), FunctionExecutionResult(content='こんにちは', name='translate_text', call_id='qpysqx3n6', is_error=False)]
---------- TextMessage (translator) ----------
안녕하세요를 영어로는 Hello, 일본어로 こんにちは라고 번역할 수 있습니다.


TaskResult(messages=[TextMessage(id='ed5de858-da47-446f-8d62-135c01b26b67', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 48, 526617, tzinfo=datetime.timezone.utc), content="'안녕하세요'를 영어와 일본어로 번역해줘", type='TextMessage'), ToolCallRequestEvent(id='e96ff829-7393-47bb-ad4a-fd6c415395d5', source='translator', models_usage=RequestUsage(prompt_tokens=793, completion_tokens=97), metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 48, 977832, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='pceyyyxe5', arguments='{"target_lang":"en","text":"안녕하세요"}', name='translate_text'), FunctionCall(id='qpysqx3n6', arguments='{"target_lang":"ja","text":"안녕하세요"}', name='translate_text')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='c6587e33-22e8-468a-9113-8140bac420cf', source='translator', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 48, 984309, tzinfo=datetime.timezone.utc), content=[Function

### 9-2. 도구 에러 처리 패턴

도구에서 예외가 발생하면 **raise하지 말고 문자열로 반환**해야 LLM이 오류를 이해하고 대처할 수 있습니다.

In [16]:
# 에러 처리가 잘 된 도구 예시
async def fetch_stock_price(ticker: str) -> str:
    """주식 종목의 현재 가격을 조회합니다. ticker는 영문 심볼(예: AAPL, TSLA)입니다."""
    # 시뮬레이션 데이터
    stock_data = {
        "AAPL": {"price": 198.50, "change": 2.3},
        "TSLA": {"price": 245.20, "change": -1.5},
        "MSFT": {"price": 415.80, "change": 0.8},
    }

    ticker = ticker.upper().strip()
    if not ticker.isalpha():
        return f"오류: '{ticker}'는 유효한 티커 형식이 아닙니다. 영문 심볼을 사용해주세요."

    if ticker not in stock_data:
        return f"오류: '{ticker}' 종목을 찾을 수 없습니다. 사용 가능: {', '.join(stock_data.keys())}"

    data = stock_data[ticker]
    return f"{ticker}: ${data['price']:.2f} ({data['change']:+.2f}%)"


# 에러 처리 테스트
error_agent = AssistantAgent(
    name="stock_agent",
    model_client=model_client,
    tools=[fetch_stock_price],
    system_message="주식 가격 조회 도구를 사용하세요. 오류가 발생하면 사용자에게 안내하세요.",
    reflect_on_tool_use=True,
)

# 존재하지 않는 티커 → 에러 메시지를 LLM이 이해하고 대처
await Console(error_agent.run_stream(task="GOOGL 주가를 알려줘"))

---------- TextMessage (user) ----------
GOOGL 주가를 알려줘
---------- ToolCallRequestEvent (stock_agent) ----------
[FunctionCall(id='ajy6j92p6', arguments='{"ticker":"GOOGL"}', name='fetch_stock_price')]
---------- ToolCallExecutionEvent (stock_agent) ----------
[FunctionExecutionResult(content="오류: 'GOOGL' 종목을 찾을 수 없습니다. 사용 가능: AAPL, TSLA, MSFT", name='fetch_stock_price', call_id='ajy6j92p6', is_error=False)]
---------- TextMessage (stock_agent) ----------
오류가 발생했습니다. 'GOOGL' 종목을 찾을 수 없습니다. 사용 가능한 종목은 AAPL, TSLA, MSFT입니다.


TaskResult(messages=[TextMessage(id='6760af0c-df2e-458a-8c85-a94ace5bf161', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 49, 300591, tzinfo=datetime.timezone.utc), content='GOOGL 주가를 알려줘', type='TextMessage'), ToolCallRequestEvent(id='55d71706-74d4-4270-b8bc-73fe91645b6b', source='stock_agent', models_usage=RequestUsage(prompt_tokens=763, completion_tokens=32), metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 49, 620315, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='ajy6j92p6', arguments='{"ticker":"GOOGL"}', name='fetch_stock_price')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='47106c27-08de-4069-99b2-7c944006d0c3', source='stock_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 49, 622639, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content="오류: 'GOOGL' 종목을 찾을 수 없습니다. 사용 가능: AAPL, TSLA, MSFT", name='fetch_stock_price', call_id='ajy6j92p6'

### 9-3. 도구 파이프라인: Agent가 여러 도구를 순차 호출

하나의 Agent에 여러 도구를 등록하면, LLM이 자율적으로 순서를 정해 호출합니다.

In [17]:
# 여러 도구를 가진 Agent → 자율적으로 파이프라인 구성
pipeline_agent = AssistantAgent(
    name="pipeline_agent",
    model_client=model_client,
    tools=[fetch_stock_price, calculate, translate_tool],
    system_message="도구를 조합하여 복잡한 요청을 처리하세요. 한국어로 답변하세요.",
    reflect_on_tool_use=True,
)

# Agent가 스스로 도구 호출 순서를 결정:
# 1) fetch_stock_price("AAPL") → 가격 확인
# 2) fetch_stock_price("TSLA") → 가격 확인
# 3) calculate("198.50 + 245.20") → 합계 계산
await Console(pipeline_agent.run_stream(
    task="AAPL과 TSLA 주가를 조회하고, 두 종목의 가격 합계를 계산해줘"
))

---------- TextMessage (user) ----------
AAPL과 TSLA 주가를 조회하고, 두 종목의 가격 합계를 계산해줘
---------- ToolCallRequestEvent (pipeline_agent) ----------
[FunctionCall(id='zkpd0jjrs', arguments='{"ticker":"AAPL"}', name='fetch_stock_price'), FunctionCall(id='rc714jngp', arguments='{"ticker":"TSLA"}', name='fetch_stock_price')]
---------- ToolCallExecutionEvent (pipeline_agent) ----------
[FunctionExecutionResult(content='AAPL: $198.50 (+2.30%)', name='fetch_stock_price', call_id='zkpd0jjrs', is_error=False), FunctionExecutionResult(content='TSLA: $245.20 (-1.50%)', name='fetch_stock_price', call_id='rc714jngp', is_error=False)]
---------- TextMessage (pipeline_agent) ----------
The total price of AAPL and TSLA is $443.70.


TaskResult(messages=[TextMessage(id='6b57cbe7-016a-4bca-a1ba-c3824e9498f5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 49, 907635, tzinfo=datetime.timezone.utc), content='AAPL과 TSLA 주가를 조회하고, 두 종목의 가격 합계를 계산해줘', type='TextMessage'), ToolCallRequestEvent(id='624d0949-bb22-4b3d-b4be-a4eb7c68df4b', source='pipeline_agent', models_usage=RequestUsage(prompt_tokens=942, completion_tokens=59), metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 50, 276103, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='zkpd0jjrs', arguments='{"ticker":"AAPL"}', name='fetch_stock_price'), FunctionCall(id='rc714jngp', arguments='{"ticker":"TSLA"}', name='fetch_stock_price')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='9cb8adb6-52e1-42b7-8da4-0d42a2eb4ec6', source='pipeline_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 12, 50, 278943, tzinfo=datetime.timezone.utc), content=[FunctionExecut

---
## 10. Termination 조건

Team 대화의 종료 시점을 결정하는 다양한 조건들을 조합하여 사용할 수 있습니다.

### 10-1. 다양한 Termination 조건 테스트

In [20]:
from autogen_agentchat.teams import RoundRobinGroupChat

from autogen_agentchat.conditions import (
    MaxMessageTermination,
    TextMentionTermination,
    TimeoutTermination,
)

# === 테스트 1: MaxMessageTermination ===
print("=== MaxMessageTermination(5) 테스트 ===")

agent_a = AssistantAgent(
    "agent_a", model_client=model_client,
    system_message="간단하게 한 문장으로만 답하세요.",
)
agent_b = AssistantAgent(
    "agent_b", model_client=model_client,
    system_message="상대방의 말에 한 문장으로 반박하세요.",
)

max_term = MaxMessageTermination(5)  # 5개 메시지 후 종료
team1 = RoundRobinGroupChat([agent_a, agent_b], termination_condition=max_term)
result = await Console(team1.run_stream(task="AI가 인간을 대체할 수 있을까?"))
print(f"\n종료 사유: {result.stop_reason}")
print(f"총 메시지 수: {len(result.messages)}")

=== MaxMessageTermination(5) 테스트 ===
---------- TextMessage (user) ----------
AI가 인간을 대체할 수 있을까?
---------- TextMessage (agent_a) ----------
 AI가 인간의 모든 영역을 대체할 수는 없습니다.
---------- TextMessage (agent_b) ----------
 공감 능력, 창의력, 그리고 복잡한 윤리적 판단과 같은 인간의 능력이 AI로 대체되기에는 아직 많은 어려움이 있습니다.
---------- TextMessage (agent_a) ----------
맞습니다.
---------- TextMessage (agent_b) ----------
인간은 감정, 직관, 경험 기반의 의사결정에서 AI보다 우월하며, 창의성과 사회적 상호작용이 필요한 분야에서는 AI가 대체할 수 없는 가치가 있습니다.

종료 사유: Maximum number of messages 5 reached, current message count: 5
총 메시지 수: 5


In [21]:
# === 테스트 2: TextMentionTermination ===
print("=== TextMentionTermination('APPROVE') 테스트 ===")

writer = AssistantAgent(
    "writer", model_client=model_client,
    system_message="주어진 주제로 짧은 글을 작성하세요.",
)
critic = AssistantAgent(
    "critic", model_client=model_client,
    system_message="글을 검토하세요. 만족스러우면 'APPROVE'라고만 응답하세요.",
)

text_term = TextMentionTermination("APPROVE") | MaxMessageTermination(10)
team2 = RoundRobinGroupChat([writer, critic], termination_condition=text_term)
result = await Console(team2.run_stream(task="커피의 장점 3가지"))
print(f"\n종료 사유: {result.stop_reason}")

=== TextMentionTermination('APPROVE') 테스트 ===
---------- TextMessage (user) ----------
커피의 장점 3가지
---------- TextMessage (writer) ----------
커피는 현대인의 일상생활에서 빼놓을 수 없는 음료입니다. 커피에는 여러 가지 장점이 있습니다. 

첫 번째, 커피는 집중력과 생산성을 높여줍니다. 커피에 포함된 카페인은 뇌 기능을 활성화하고, 집중력을 향상하는 데 도움을 줍니다. 따라서, 공부나 일을 할 때 커피를 마시면 더 효율적으로 작업할 수 있습니다.

두 번째, 커피는 심혈관 건강을 개선하는 데 도움이 됩니다. 적당한 커피 섭취는 심장병, 뇌졸중, 당뇨병 등의 위험을 감소시키는 효과가 있습니다. 

세 번째, 커피는 운동 성능을 향상하는 데 도움이 됩니다. 카페인은 근육의 수축력을 증가시키고, 지구력을 높여주는 효과가 있습니다. 따라서, 운동 전 커피를 마시면 더 좋은 성과를 낼 수 있습니다. 

이처럼 커피는 우리의 삶에 여러모로 도움을 주는 음료입니다. 하지만, 과도한 커피 섭취는 오히려 건강에 해로울 수 있으므로, 적절한 양을 섭취하는 것이 중요합니다.
---------- TextMessage (critic) ----------
APPROVE

종료 사유: Text 'APPROVE' mentioned


### 10-2. OR / AND 조합

- `|` (OR): 하나라도 충족되면 종료
- `&` (AND): 모두 충족되어야 종료

In [22]:
# === OR 조합: 안전 장치 패턴 ===
# 정상 종료(APPROVE) 또는 메시지 상한(20) 또는 시간 초과(60초)
safe_termination = (
    TextMentionTermination("APPROVE")
    | MaxMessageTermination(20)
    | TimeoutTermination(timeout_seconds=60)
)
print(f"OR 조합 생성 완료: {safe_termination}")

# === AND 조합: 엄격한 종료 ===
# 메시지 10개 이상 AND 'DONE' 키워드 언급 → 둘 다 충족해야 종료
strict_termination = (
    MaxMessageTermination(10)
    & TextMentionTermination("DONE")
)
print(f"AND 조합 생성 완료: {strict_termination}")

# AND + OR 혼합
# (APPROVE가 나오면 즉시 종료) 또는 (메시지 10개 이상 AND DONE)
mixed_termination = (
    TextMentionTermination("APPROVE")
    | (MaxMessageTermination(10) & TextMentionTermination("DONE"))
)
print(f"혼합 조합 생성 완료: {mixed_termination}")

OR 조합 생성 완료: <autogen_agentchat.base._termination.OrTerminationCondition object at 0x118cce520>
AND 조합 생성 완료: <autogen_agentchat.base._termination.AndTerminationCondition object at 0x1189e6900>
혼합 조합 생성 완료: <autogen_agentchat.base._termination.OrTerminationCondition object at 0x118cce650>


### 10-3. Team 상태 관리: reset과 대화 재개

In [23]:
# Team 상태 관리 데모
simple_agent = AssistantAgent(
    "simple", model_client=model_client,
    system_message="간결하게 한국어로 답하세요.",
)

team3 = RoundRobinGroupChat(
    [simple_agent],
    max_turns=1,
)

# 1차 실행
print("=== 1차 실행 ===")
result1 = await Console(team3.run_stream(task="내 이름은 민수야. 기억해줘."))

# 2차 실행 (이전 대화 이어감 - reset 안 함)
print("\n=== 2차 실행 (대화 이어감) ===")
result2 = await Console(team3.run_stream(task="내 이름이 뭐라고 했지?"))

# reset 후 3차 실행 (새로운 대화)
await team3.reset()
print("\n=== 3차 실행 (reset 후 새 대화) ===")
result3 = await Console(team3.run_stream(task="내 이름이 뭐라고 했지?"))

=== 1차 실행 ===
---------- TextMessage (user) ----------
내 이름은 민수야. 기억해줘.
---------- TextMessage (simple) ----------
내가 이름을 기억하도록 할게. 민수야.

=== 2차 실행 (대화 이어감) ===
---------- TextMessage (user) ----------
내 이름이 뭐라고 했지?
---------- TextMessage (simple) ----------
민수야.

=== 3차 실행 (reset 후 새 대화) ===
---------- TextMessage (user) ----------
내 이름이 뭐라고 했지?
---------- TextMessage (simple) ----------
저는 당신의 이름을 모릅니다. 이 대화는 처음입니다.


---
## 11. 에이전트 상태 저장과 복원

`save_state()` / `load_state()`로 Team과 Agent의 상태를 JSON으로 저장하고, 새 세션에서 복원할 수 있습니다.

### 11-1. Team 상태 저장/복원

In [24]:
import json

# 1. Team 생성 및 첫 실행
writer_s = AssistantAgent(
    "writer", model_client=model_client,
    system_message="주어진 주제로 짧은 에세이를 작성하세요.",
)
critic_s = AssistantAgent(
    "critic", model_client=model_client,
    system_message="에세이를 검토하고 피드백을 주세요. 만족스러우면 'APPROVE'라고 응답하세요.",
)

term = TextMentionTermination("APPROVE") | MaxMessageTermination(6)
team_save = RoundRobinGroupChat([writer_s, critic_s], termination_condition=term)

print("=== 1단계: 첫 실행 ===")
result = await Console(team_save.run_stream(task="AI 윤리에 대한 짧은 에세이를 써줘"))

# 2. 상태 저장
state = await team_save.save_state()
with open("team_state.json", "w") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)
print(f"\n상태 저장 완료! (파일 크기: {len(json.dumps(state))} bytes)")

=== 1단계: 첫 실행 ===
---------- TextMessage (user) ----------
AI 윤리에 대한 짧은 에세이를 써줘
---------- TextMessage (writer) ----------
AI 윤리는 인공지능 기술의 발전과 활용에 따른 윤리적 문제를 다루는 분야입니다. AI 기술은 우리의 삶에 큰 변화를 가져오고 있지만, 동시에 여러 가지 윤리적 문제를 제기하고 있습니다.

가장 큰 문제 중 하나는 데이터의 편향성입니다. AI는 데이터를 기반으로 학습하고 의사 결정을 내리기 때문에, 데이터에 편향이 있으면 AI의 결과도 편향될 수 있습니다. 예를 들어, 얼굴 인식 시스템이 특정 인종이나 성별에 대해 편향된 결과를 내놓을 수 있습니다.

또 다른 문제는 AI의 투명성입니다. AI가 어떻게 의사 결정을 내리는지 이해하기 어렵기 때문에, 우리는 AI의 결과에 대해 의문을 가질 수 있습니다. 예를 들어, 자율 주행 자동차가 사고를 일으켰을 때, 우리는 AI가 어떻게 그런 결정을 내렸는지 알고 싶을 것입니다.

또한 AI의 책임성에 대한 문제도 있습니다. AI가 잘못된 결정을 내렸을 때, 누가 책임을 져야 하는가? 개발사? 사용자? 이러한 문제들은 AI 윤리에서 중요한 이슈들입니다.

마지막으로, AI의 활용에 따른 일자리 감소 문제도 있습니다. AI가 많은 작업을 자동화하면서, 일부 직업이 사라질 수 있습니다. 우리는 이러한 변화에 대비하고, 새로운 직업 기회를 창출해야 합니다.

AI 윤리는 우리가 AI 기술을 발전시키고 활용하는 데 있어 중요한 고려 사항입니다. 우리는 AI의 잠재적인 위험과 기회를 균형 있게 고려하고, 윤리적인 원칙을 기반으로 AI 기술을 개발하고 활용해야 합니다.
---------- TextMessage (critic) ----------
APPROVE

However, I do have some minor suggestions for improvement:

* Consider adding a clear the

In [25]:
# 3. 새 Team 인스턴스에 상태 복원 (새 세션 시뮬레이션)
writer_s2 = AssistantAgent(
    "writer", model_client=model_client,
    system_message="주어진 주제로 짧은 에세이를 작성하세요.",
)
critic_s2 = AssistantAgent(
    "critic", model_client=model_client,
    system_message="에세이를 검토하고 피드백을 주세요. 만족스러우면 'APPROVE'라고 응답하세요.",
)

term2 = TextMentionTermination("APPROVE") | MaxMessageTermination(6)
team_restored = RoundRobinGroupChat([writer_s2, critic_s2], termination_condition=term2)

# 상태 파일에서 복원
with open("team_state.json", "r") as f:
    saved_state = json.load(f)
await team_restored.load_state(saved_state)

# 4. 이전 대화를 이어서 진행
print("=== 2단계: 상태 복원 후 대화 재개 ===")
result = await Console(team_restored.run_stream(task="결론 부분을 좀 더 강하게 수정해줘"))
print(f"\n종료 사유: {result.stop_reason}")

=== 2단계: 상태 복원 후 대화 재개 ===
---------- TextMessage (user) ----------
결론 부분을 좀 더 강하게 수정해줘
---------- TextMessage (writer) ----------
Here is the rewritten essay with a stronger conclusion:

AI 윤리는 인공지능 기술의 발전과 활용에 따른 윤리적 문제를 다루는 분야입니다. 최근 AI 기술의 급속한 발전으로 인해 우리는 AI의 잠재적인 위험과 기회를 균형 있게 고려해야 합니다.

가장 큰 문제 중 하나는 데이터의 편향성입니다. AI는 데이터를 기반으로 학습하고 의사 결정을 내리기 때문에, 데이터에 편향이 있으면 AI의 결과도 편향될 수 있습니다. 예를 들어, 얼굴 인식 시스템이 특정 인종이나 성별에 대해 편향된 결과를 내놓을 수 있습니다. 이러한 문제는 공정성과 정확성을 저해할 수 있습니다.

또 다른 문제는 AI의 투명성입니다. AI가 어떻게 의사 결정을 내리는지 이해하기 어렵기 때문에, 우리는 AI의 결과에 대해 의문을 가질 수 있습니다. 예를 들어, 자율 주행 자동차가 사고를 일으켰을 때, 우리는 AI가 어떻게 그런 결정을 내렸는지 알고 싶을 것입니다.

또한 AI의 책임성에 대한 문제도 있습니다. AI가 잘못된 결정을 내렸을 때, 누가 책임을 져야 하는가? 개발사? 사용자? 이러한 문제들은 AI 윤리에서 중요한 이슈들입니다.

마지막으로, AI의 활용에 따른 일자리 감소 문제도 있습니다. AI가 많은 작업을 자동화하면서, 일부 직업이 사라질 수 있습니다. 우리는 이러한 변화에 대비하고, 새로운 직업 기회를 창출해야 합니다.

결론적으로, AI 윤리는 단순히 기술적인 문제가 아니라 사회적, 경제적, 윤리적 문제를 포괄하는 중요한 이슈입니다. 우리는 AI의 발전과 활용에 있어 윤리적인 원칙을 기반으로 한 책임감 있는 접근을 취해야 하며, 이를 통해 AI가 인간의 삶을 개선하고 사회에 긍정적인 영향을 미칠 수 있도록 

### 11-2. 개별 Agent 상태 저장/복원

In [26]:
# 개별 Agent 상태 저장
agent_orig = AssistantAgent(
    "memory_agent", model_client=model_client,
    system_message="간결하게 답하세요.",
)

# 대화 기록 생성
team_a = RoundRobinGroupChat([agent_orig], max_turns=1)
await Console(team_a.run_stream(task="내 이름은 영희, 좋아하는 색은 파란색이야."))
await Console(team_a.run_stream(task="내 취미는 독서야."))

# Agent 상태 저장
agent_state = await agent_orig.save_state()
print(f"Agent 상태 키: {list(agent_state.keys())}")

# 새 Agent 인스턴스에 복원
agent_new = AssistantAgent(
    "memory_agent", model_client=model_client,
    system_message="간결하게 답하세요.",
)
await agent_new.load_state(agent_state)

# 복원된 Agent로 확인
team_b = RoundRobinGroupChat([agent_new], max_turns=1)
print("\n=== 복원된 Agent에게 질문 ===")
await Console(team_b.run_stream(task="내 이름, 좋아하는 색, 취미가 뭐라고 했지?"))

---------- TextMessage (user) ----------
내 이름은 영희, 좋아하는 색은 파란색이야.
---------- TextMessage (memory_agent) ----------
안녕, 영희!
---------- TextMessage (user) ----------
내 취미는 독서야.
---------- TextMessage (memory_agent) ----------
좋아하는 색은 파란색이고 취미는 독서가 좋네요!
Agent 상태 키: ['type', 'version', 'llm_context']

=== 복원된 Agent에게 질문 ===
---------- TextMessage (user) ----------
내 이름, 좋아하는 색, 취미가 뭐라고 했지?
---------- TextMessage (memory_agent) ----------
이름은 영희, 좋아하는 색은 파란색, 취미는 독서입니다.


TaskResult(messages=[TextMessage(id='70d57d28-1493-4e88-bdfe-0049d681a9c5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 15, 14, 306153, tzinfo=datetime.timezone.utc), content='내 이름, 좋아하는 색, 취미가 뭐라고 했지?', type='TextMessage'), TextMessage(id='5d9c1fd0-eb41-45de-9227-824c01201ea1', source='memory_agent', models_usage=RequestUsage(prompt_tokens=102, completion_tokens=21), metadata={}, created_at=datetime.datetime(2026, 4, 21, 7, 15, 14, 902193, tzinfo=datetime.timezone.utc), content='이름은 영희, 좋아하는 색은 파란색, 취미는 독서입니다.', type='TextMessage')], stop_reason='Maximum number of turns 1 reached.')

---
## 12. 디버깅과 모니터링

`TaskResult` 객체를 분석하여 대화 흐름, 종료 사유, 토큰 사용량을 추적합니다.

### 12-1. 실행 결과(TaskResult) 분석

In [27]:
# 분석용 Team 실행
planner_d = AssistantAgent(
    "Planner",
    description="작업을 계획하는 에이전트.",
    model_client=model_client,
    system_message="작업을 3단계로 분해하세요.",
)
executor_d = AssistantAgent(
    "Executor",
    description="계획을 실행하는 에이전트.",
    model_client=model_client,
    system_message="계획을 실행하고 결과를 보고하세요. 완료 시 'DONE'이라고 적으세요.",
)

term_d = TextMentionTermination("DONE") | MaxMessageTermination(6)
team_d = SelectorGroupChat(
    [planner_d, executor_d],
    model_client=model_client,
    termination_condition=term_d,
)

result = await team_d.run(task="Python으로 피보나치 함수를 작성하는 계획을 세워줘")

# === 결과 분석 ===
print("=" * 60)
print("대화 흐름 분석")
print("=" * 60)

for i, msg in enumerate(result.messages):
    content_preview = msg.content[:80] if isinstance(msg.content, str) else str(msg.content)[:80]
    print(f"  [{i}] {msg.source:12s} | {type(msg).__name__:20s} | {content_preview}...")

print(f"\n총 메시지 수: {len(result.messages)}")
print(f"종료 사유: {result.stop_reason}")

대화 흐름 분석
  [0] user         | TextMessage          | Python으로 피보나치 함수를 작성하는 계획을 세워줘...
  [1] Planner      | TextMessage          | 1. **문제 이해**: 
   - 피보나치 수열은 앞의 두 숫자의 합으로 만들어지는 수열입니다. 
   - 예를 들어, 첫 두 숫자가 0과 1...
  [2] Executor     | TextMessage          | ### 피보나치 함수 작성 계획

#### 1. 문제 이해
- **피보나치 수열**: 앞의 두 숫자의 합으로 만들어지는 수열.  
  예: 첫 ...

총 메시지 수: 3
종료 사유: Text 'DONE' mentioned


### 12-2. 토큰 사용량 추적

In [28]:
# 토큰 사용량 분석
print("=" * 60)
print("토큰 사용량 분석")
print("=" * 60)

total_prompt = 0
total_completion = 0

for msg in result.messages:
    if hasattr(msg, "models_usage") and msg.models_usage:
        usage = msg.models_usage
        prompt_tokens = usage.prompt_tokens
        completion_tokens = usage.completion_tokens
        total_prompt += prompt_tokens
        total_completion += completion_tokens
        print(f"  [{msg.source:12s}] 입력: {prompt_tokens:,}  출력: {completion_tokens:,}")

print(f"\n총 입력 토큰: {total_prompt:,}")
print(f"총 출력 토큰: {total_completion:,}")
print(f"총 토큰: {total_prompt + total_completion:,}")

# GPT-4o 기준 비용 추정 (2024년 기준)
cost = (total_prompt * 2.5 / 1_000_000) + (total_completion * 10 / 1_000_000)
print(f"예상 비용 (GPT-4o): ${cost:.4f}")

토큰 사용량 분석
  [Planner     ] 입력: 38  출력: 379
  [Executor    ] 입력: 429  출력: 564

총 입력 토큰: 467
총 출력 토큰: 943
총 토큰: 1,410
예상 비용 (GPT-4o): $0.0106


### 12-3. 로깅 설정

AutoGen 내부 로그를 활성화하여 Agent 선택, 도구 호출 등의 상세 과정을 확인합니다.

In [29]:
import logging

# AutoGen 로깅 활성화
logging.basicConfig(level=logging.WARNING)
logging.getLogger("autogen_agentchat").setLevel(logging.DEBUG)

# 로그가 활성화된 상태에서 실행 → 내부 동작을 상세히 확인
debug_agent = AssistantAgent(
    "debug_agent", model_client=model_client,
    tools=[get_weather],
    system_message="도구를 사용하세요.",
    reflect_on_tool_use=True,
)

debug_team = RoundRobinGroupChat([debug_agent], max_turns=1)
await Console(debug_team.run_stream(task="서울 날씨 알려줘"))

# 로깅 원복
logging.getLogger("autogen_agentchat").setLevel(logging.WARNING)

---------- TextMessage (user) ----------
서울 날씨 알려줘


DEBUG:autogen_agentchat.events:id='d07f185c-62ee-40a7-a732-3fc3aa7e66eb' source='debug_agent' models_usage=RequestUsage(prompt_tokens=731, completion_tokens=36) metadata={} created_at=datetime.datetime(2026, 4, 21, 7, 15, 23, 902118, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='jbbzpw24v', arguments='{"city":"어경"}', name='get_weather')] type='ToolCallRequestEvent'


---------- ToolCallRequestEvent (debug_agent) ----------


DEBUG:autogen_agentchat.events:id='f469b798-d3cf-4faf-af07-932b9eb44f40' source='debug_agent' models_usage=None metadata={} created_at=datetime.datetime(2026, 4, 21, 7, 15, 23, 903868, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='어경: 날씨 데이터 없음', name='get_weather', call_id='jbbzpw24v', is_error=False)] type='ToolCallExecutionEvent'


[FunctionCall(id='jbbzpw24v', arguments='{"city":"어경"}', name='get_weather')]
---------- ToolCallExecutionEvent (debug_agent) ----------
[FunctionExecutionResult(content='어경: 날씨 데이터 없음', name='get_weather', call_id='jbbzpw24v', is_error=False)]
---------- TextMessage (debug_agent) ----------
[
  {
    "name": "get_weather",
    "parameters": {
      "city": "\uc5b4\uaac8"
    }
  }
]


---
## 13. 통합 실습: 코드 리뷰 에이전트 팀

4개의 전문 Agent(Architect, SecurityReviewer, CodeOptimizer, Summarizer)가 협업하여 코드를 리뷰합니다.

### 13-1. 도구 정의

In [30]:
async def count_lines(code: str) -> str:
    """코드의 줄 수, 함수 수, 클래스 수를 분석합니다."""
    lines = code.strip().split("\n")
    func_count = sum(1 for line in lines if line.strip().startswith("def "))
    class_count = sum(1 for line in lines if line.strip().startswith("class "))
    return (
        f"분석 결과:\n"
        f"  - 총 줄 수: {len(lines)}\n"
        f"  - 함수 수: {func_count}\n"
        f"  - 클래스 수: {class_count}"
    )


async def check_security_patterns(code: str) -> str:
    """코드에서 보안 취약점 패턴을 검사합니다."""
    issues = []
    if "eval(" in code:
        issues.append("[심각] eval() 사용 감지 - 코드 인젝션 위험")
    if "exec(" in code:
        issues.append("[심각] exec() 사용 감지 - 코드 인젝션 위험")
    if "password" in code.lower() and "=" in code:
        issues.append("[심각] 하드코딩된 비밀번호 의심")
    if "SELECT" in code and "+" in code:
        issues.append("[심각] SQL 인젝션 가능성 - 문자열 연결로 쿼리 생성")
    if "import os" in code and "system(" in code:
        issues.append("[경고] os.system() 사용 - 명령어 인젝션 위험")
    if "pickle.load" in code:
        issues.append("[경고] pickle.load() 사용 - 역직렬화 공격 위험")

    if issues:
        return "보안 취약점 발견:\n" + "\n".join(f"  {i}" for i in issues)
    return "주요 보안 패턴 이슈 없음"


# 도구 테스트
test_code = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    result = eval(db.execute(query))
    password = "admin123"
    return result
"""

print(await count_lines(test_code))
print()
print(await check_security_patterns(test_code))

분석 결과:
  - 총 줄 수: 5
  - 함수 수: 1
  - 클래스 수: 0

보안 취약점 발견:
  [심각] eval() 사용 감지 - 코드 인젝션 위험
  [심각] 하드코딩된 비밀번호 의심
  [심각] SQL 인젝션 가능성 - 문자열 연결로 쿼리 생성


### 13-2. 코드 리뷰 Agent 팀 구성

In [31]:
architect = AssistantAgent(
    "Architect",
    description="코드의 구조, 설계 패턴, 아키텍처를 분석하는 에이전트.",
    model_client=model_client,
    tools=[count_lines],
    system_message="""코드의 구조적 품질을 분석하세요:
    - SOLID 원칙 준수 여부
    - 함수/클래스 설계
    - 네이밍 컨벤션
    - 관심사 분리(SoC)
    count_lines 도구로 코드 통계를 먼저 확인하세요.""",
)

security_reviewer = AssistantAgent(
    "SecurityReviewer",
    description="코드의 보안 취약점을 검토하는 에이전트.",
    model_client=model_client,
    tools=[check_security_patterns],
    system_message="""코드의 보안 취약점을 검토하세요:
    - check_security_patterns 도구로 자동 검사를 먼저 수행하세요
    - 입력 검증, 인젝션 위험, 인증/인가, 민감 데이터 노출을 확인하세요""",
)

optimizer = AssistantAgent(
    "CodeOptimizer",
    description="코드의 성능과 효율성을 분석하고 개선안을 제시하는 에이전트.",
    model_client=model_client,
    system_message="""코드의 성능 최적화 방안을 제시하세요:
    - 시간/공간 복잡도
    - 불필요한 연산 제거
    - 알고리즘 개선 기회""",
)

review_summarizer = AssistantAgent(
    "Summarizer",
    description="모든 리뷰 의견을 종합하여 최종 코드 리뷰를 작성하는 에이전트.",
    model_client=model_client,
    system_message="""다른 에이전트들의 리뷰를 종합하여 최종 코드 리뷰를 작성하세요.
    형식:
    ## 종합 코드 리뷰
    ### 심각도 상
    - ...
    ### 심각도 중
    - ...
    ### 심각도 하
    - ...
    ### 개선된 코드 제안
    최종 리뷰 작성 완료 시 반드시 'REVIEW_COMPLETE'라고 마지막에 적으세요.""",
)

print("Agent 4개 생성 완료: Architect, SecurityReviewer, CodeOptimizer, Summarizer")

Agent 4개 생성 완료: Architect, SecurityReviewer, CodeOptimizer, Summarizer


### 13-3. 코드 리뷰 실행

In [32]:
review_termination = TextMentionTermination("REVIEW_COMPLETE") | MaxMessageTermination(20)

review_team = SelectorGroupChat(
    [architect, security_reviewer, optimizer, review_summarizer],
    model_client=model_client,
    termination_condition=review_termination,
)

# 리뷰할 코드
code_to_review = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    result = eval(db.execute(query))
    password = "admin123"
    return result

def process_data(data):
    output = []
    for i in range(len(data)):
        for j in range(len(data)):
            if data[i] == data[j] and i != j:
                output.append(data[i])
    return output
"""

result = await Console(review_team.run_stream(
    task=f"다음 Python 코드를 리뷰해주세요:\n```python\n{code_to_review}\n```"
))

---------- TextMessage (user) ----------
다음 Python 코드를 리뷰해주세요:
```python

def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    result = eval(db.execute(query))
    password = "admin123"
    return result

def process_data(data):
    output = []
    for i in range(len(data)):
        for j in range(len(data)):
            if data[i] == data[j] and i != j:
                output.append(data[i])
    return output

```
---------- ToolCallRequestEvent (SecurityReviewer) ----------
[FunctionCall(id='5xn5veamg', arguments='{"code":"def get_user(user_id):\\n    query = \\"SELECT * FROM users WHERE id = \\" + str(user_id)\\n    result = eval(db.execute(query))\\n    password = \\"admin123\\"\\n    return result\\n\\ndef process_data(data):\\n    output = []\\n    for i in range(len(data)):\\n        for j in range(len(data)):\\n            if data[i] == data[j] and i != j:\\n                output.append(data[i])\\n    return output\\n"}', name='check_security

In [33]:
# 코드 리뷰 결과 분석
print("=" * 60)
print("코드 리뷰 참여 Agent 분석")
print("=" * 60)

from collections import Counter
agent_counts = Counter(msg.source for msg in result.messages if msg.source != "user")
for agent_name, count in agent_counts.most_common():
    print(f"  {agent_name}: {count}회 발언")

print(f"\n총 메시지: {len(result.messages)}개")
print(f"종료 사유: {result.stop_reason}")

코드 리뷰 참여 Agent 분석
  SecurityReviewer: 3회 발언
  Architect: 3회 발언
  CodeOptimizer: 1회 발언
  Summarizer: 1회 발언

총 메시지: 9개
종료 사유: Text 'REVIEW_COMPLETE' mentioned


---
## 14. 실전 팁과 베스트 프랙티스

### 14-1. selector_func으로 커스텀 라우팅

SelectorGroupChat에서 `selector_func`을 사용하면 LLM 기반 선택을 보완하거나 오버라이드할 수 있습니다.

In [34]:
from typing import Sequence
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage

# 커스텀 선택 함수: 항상 Planner를 먼저 거치도록 라우팅
def planner_first_selector(
    messages: Sequence[BaseAgentEvent | BaseChatMessage],
) -> str | None:
    """마지막 발언자가 Planner가 아니면 Planner로 보내고,
    Planner면 None(LLM 선택)으로 폴백"""
    if not messages:
        return "Planner"
    last_source = messages[-1].source
    if last_source == "Planner":
        return None  # LLM이 다음 Agent 선택
    return "Planner"  # 항상 Planner를 거침


planner_f = AssistantAgent(
    "Planner",
    description="작업을 계획하는 에이전트.",
    model_client=model_client,
    system_message="작업을 분석하고 다음 단계를 지시하세요. 완료 시 'COMPLETE'라고 적으세요.",
)
worker_f = AssistantAgent(
    "Worker",
    description="실제 작업을 수행하는 에이전트.",
    model_client=model_client,
    system_message="Planner의 지시에 따라 작업을 수행하세요.",
)

term_f = TextMentionTermination("COMPLETE") | MaxMessageTermination(8)
team_f = SelectorGroupChat(
    [planner_f, worker_f],
    model_client=model_client,
    termination_condition=term_f,
    selector_func=planner_first_selector,
)

result = await Console(team_f.run_stream(task="'Hello World'를 5개 언어로 번역해줘"))

# 발언 순서 확인 (항상 Planner를 거침)
print("\n발언 순서:")
for msg in result.messages:
    print(f"  → {msg.source}")

---------- TextMessage (user) ----------
'Hello World'를 5개 언어로 번역해줘
---------- TextMessage (Planner) ----------
다음은 'Hello World'를 5개 언어로 번역한 결과입니다.

1. 영어: Hello World
2. 프랑스어: Bonjour le monde
3. 스페인어: Hola mundo
4. 독일어: Hallo Welt
5. 중국어: (nǐ hǎo, shì jiè)

COMPLETE

발언 순서:
  → user
  → Planner


### 14-2. Writer-Critic Reflection 패턴 (RoundRobin)

가장 대표적인 RoundRobin 활용 패턴입니다. Writer가 글을 쓰고 Critic이 피드백하는 반복 구조.

In [35]:
# Writer-Critic Reflection 패턴
reflection_writer = AssistantAgent(
    "writer",
    model_client=model_client,
    system_message="""당신은 기술 블로그 작가입니다.
    주어진 주제로 짧은 글(200자 내외)을 작성하세요.
    Critic의 피드백이 있으면 반영하여 수정하세요.""",
)

reflection_critic = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="""당신은 기술 글 편집자입니다. 글을 검토하고 구체적인 피드백을 제공하세요.
    평가 기준: 정확성, 가독성, 구체성
    만족스러우면 'APPROVE'라고만 응답하세요.
    최대 2번까지만 피드백하고, 3번째에는 반드시 APPROVE하세요.""",
)

reflection_term = TextMentionTermination("APPROVE") | MaxMessageTermination(8)
reflection_team = RoundRobinGroupChat(
    [reflection_writer, reflection_critic],
    termination_condition=reflection_term,
)

result = await Console(reflection_team.run_stream(
    task="Python의 async/await가 필요한 이유를 설명하는 짧은 글을 써줘"
))

# 몇 라운드만에 완료되었는지 확인
rounds = sum(1 for msg in result.messages if msg.source == "writer")
print(f"\n작성 라운드: {rounds}회")
print(f"종료 사유: {result.stop_reason}")

---------- TextMessage (user) ----------
Python의 async/await가 필요한 이유를 설명하는 짧은 글을 써줘
---------- TextMessage (writer) ----------
### 파이썬의 async/await가 필요한 이유

비동기 프로그래밍은 현대 소프트웨어 개발에서 중요한 개념입니다. 파이썬의 async/await는 비동기 프로그래밍을 더욱 쉽게 만들어주는 강력한 도구입니다.

**문제점:** 전통적인 동기식 프로그래밍 방식에서는 코드가 순차적으로 실행되며, 하나의 작업이 완료되어야 다음 작업이 시작됩니다. 이러한 방식은 I/O 작업이 많은 프로그램에서 성능 저하를 초래할 수 있습니다.

**해결책:** async/await를 사용하면 비동기 작업을 쉽게 작성할 수 있습니다. async 키워드로 정의된 함수는 비동기 함수이며, await 키워드로 비동기 작업을 기다릴 수 있습니다. 이를 통해 프로그램은 다른 작업을 수행하면서 I/O 작업이 완료되기를 기다릴 수 있습니다.

**예시:**
```python
import asyncio

async def fetch_data():
    await asyncio.sleep(2)  # 2초 동안 대기
    return "데이터를 가져왔습니다."

async def main():
    task = asyncio.create_task(fetch_data())
    print("다른 작업을 수행합니다.")
    result = await task
    print(result)

asyncio.run(main())
```
async/await를 사용하면 비동기 프로그래밍이 더욱 쉬워지고, 프로그램의 성능을 향상시킬 수 있습니다.
---------- TextMessage (critic) ----------
### 피드백

*   정확성: async/await의 사용 이유와 예시를 정확하게 제시했습니다.
*   가독성: 코드 예시가 포함되어 있어 이해하기 쉬우나

### 14-3. 전체 패턴 비교 실험

동일한 작업을 RoundRobin, SelectorGroupChat, Swarm 세 패턴으로 실행하고 결과를 비교합니다.

In [36]:
import time

task = "Python과 JavaScript의 장단점을 비교해줘"

# === 패턴 1: RoundRobin ===
rr_a = AssistantAgent("analyst", model_client=model_client,
    system_message="기술 비교 분석을 수행하세요.")
rr_b = AssistantAgent("reviewer", model_client=model_client,
    system_message="분석을 검토하세요. 만족스러우면 'DONE'이라고 응답하세요.")
rr_term = TextMentionTermination("DONE") | MaxMessageTermination(6)
rr_team = RoundRobinGroupChat([rr_a, rr_b], termination_condition=rr_term)

t0 = time.time()
rr_result = await rr_team.run(task=task)
rr_time = time.time() - t0

# === 패턴 2: SelectorGroupChat ===
sg_a = AssistantAgent("PythonExpert", description="Python 전문가.",
    model_client=model_client, system_message="Python 관점에서 분석하세요.")
sg_b = AssistantAgent("JSExpert", description="JavaScript 전문가.",
    model_client=model_client, system_message="JavaScript 관점에서 분석하세요.")
sg_c = AssistantAgent("Summarizer", description="결과를 종합하는 에이전트.",
    model_client=model_client,
    system_message="두 전문가의 의견을 종합하세요. 완료 시 'DONE'이라고 적으세요.")
sg_term = TextMentionTermination("DONE") | MaxMessageTermination(8)
sg_team = SelectorGroupChat([sg_a, sg_b, sg_c], model_client=model_client,
    termination_condition=sg_term)

t0 = time.time()
sg_result = await sg_team.run(task=task)
sg_time = time.time() - t0

# === 결과 비교 ===
print("=" * 60)
print("패턴별 결과 비교")
print("=" * 60)
print(f"{'패턴':<25} {'메시지 수':>8} {'소요 시간':>10}")
print("-" * 60)
print(f"{'RoundRobinGroupChat':<25} {len(rr_result.messages):>8} {rr_time:>9.1f}s")
print(f"{'SelectorGroupChat':<25} {len(sg_result.messages):>8} {sg_time:>9.1f}s")

패턴별 결과 비교
패턴                           메시지 수      소요 시간
------------------------------------------------------------
RoundRobinGroupChat              3       1.8s
SelectorGroupChat                3       2.0s


### 14-4. 정리: 패턴 선택 가이드

| 패턴 | 적합한 작업 | 복잡도 |
|:---:|:---:|:---:|
| **RoundRobin** | Writer-Critic, 순차 피드백 | 낮음 |
| **SelectorGroupChat** | 전문가 협업, 동적 라우팅 | 중간 |
| **Swarm** | 워크플로우, 사용자 핸드오프 | 높음 |

In [37]:
# 임시 파일 정리
import os
if os.path.exists("team_state.json"):
    os.remove("team_state.json")
    print("team_state.json 정리 완료")

team_state.json 정리 완료
